# Vigenere - why absolute position encoding fails

PATE cannot be built with learned absolute positional encoding: it does not
converge, so the failure cannot be examined there. This notebook uses a variant
that supplies more structure than PATE does - cycle-position embeddings indexed
by (position mod d) instead of a single period vector - which converges under
absolute encoding on the narrower key range d in [3, 12]. What it still cannot
do is then attributable to the encoding alone.

Two models are trained here, differing only in the position encoding:

| model | position encoding | period conditioning |
|---|---|---|
| `model_cycle_la`   | learned absolute | per-cycle embeddings |
| `model_cycle_rope` | rotary | per-cycle embeddings |

A third, `model_cycle_sin`, replaces the learned absolute encoding with a fixed
sinusoidal one whose components span the full range of periods. It is the
control for the frequency-content explanation.

The outputs are the frequency spectrum of the learned position embeddings and
the blind period estimation confusion matrix.

In [ ]:
import os, sys
from pathlib import Path

def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if in_colab():
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = Path('/content/drive/MyDrive/cryptanalysis-ai')
    %pip install -q pytorch-lightning torchmetrics matplotlib
else:
    REPO_DIR = Path.cwd()
    if REPO_DIR.name == 'notebooks':
        REPO_DIR = REPO_DIR.parent

os.environ['CRYPT_DATA_DIR'] = str(REPO_DIR / 'data')
os.environ['CRYPT_CKPT_DIR'] = str(REPO_DIR / 'checkpoints')
os.environ['CRYPT_LOG_DIR']  = str(REPO_DIR / 'tb_logs')
sys.path.insert(0, str(REPO_DIR / 'src'))
print('repo:', REPO_DIR)


In [ ]:
VARIANT = "cycle_la"    # "cycle_la" | "cycle_rope" | "cycle_sin"

import importlib
import torch, config
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.loggers import TensorBoardLogger

SPECS = {
    "cycle_la":   ("models.vigenere.model_cycle_la",   "vigenere_cycle_la"),
    "cycle_rope": ("models.vigenere.model_cycle_rope", "vigenere_cycle_rope"),
    "cycle_sin":  ("models.vigenere.model_cycle_sin",  "vigenere_cycle_sin"),
}
module, CKPT_NAME = SPECS[VARIANT]
mod = importlib.import_module(module)
VigenereSolver, UNKNOWN_KEY = mod.VigenereSolver, mod.UNKNOWN_KEY

pl.seed_everything(42, workers=True)
print(f"variant={VARIANT}  checkpoints -> {CKPT_NAME}")


In [ ]:
from models.vigenere.data_module import VigenereDataModule

MIN_K_LEN, MAX_K_LEN, BATCH_SIZE = 3, 12, 128

dm = VigenereDataModule(
    text_path=str(config.data_file("final_dataset_shuffled.txt")),
    alphabet=config.ALPHABET,
    batch_size=BATCH_SIZE,
    min_key_len=MIN_K_LEN,
    max_key_len=MAX_K_LEN,
)
dm.setup()

model = VigenereSolver(
    vocab_size=dm.dataset.crypto_vocab_size,
    min_key_len=MIN_K_LEN, max_key_len=MAX_K_LEN,
    d_model=256, nhead=8, num_layers=6, lr=1e-3,
)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")


In [ ]:
ckpt_dir, log_dir = config.model_paths(CKPT_NAME)

trainer = pl.Trainer(
    accelerator="auto", devices=1, max_epochs=230, min_epochs=80,
    callbacks=[
        ModelCheckpoint(dirpath=str(ckpt_dir),
                        filename=VARIANT + "-best-keylen-{epoch:02d}-{val_keylen_acc:.3f}-{val_acc:.3f}",
                        save_top_k=2, save_last=True, monitor="val_keylen_acc", mode="max"),
        EarlyStopping(monitor="val_keylen_acc", patience=40, mode="max", strict=False),
    ],
    logger=TensorBoardLogger(str(log_dir), name=CKPT_NAME),
    gradient_clip_val=1.0, log_every_n_steps=10, precision="32-true",
)
trainer.fit(model, datamodule=dm)


## Blind period estimation

In [ ]:
import numpy as np
from models.vigenere import inference as vinf, evaluation as ev

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
best = sorted((config.CHECKPOINT_DIR / CKPT_NAME).glob("*best-keylen*.ckpt"))[-1]
m = VigenereSolver.load_from_checkpoint(str(best), map_location=DEVICE)
m.to(DEVICE).eval(); m.freeze()
print("checkpoint:", best.name)

char2idx, idx2char = vinf.build_vocab(config.ALPHABET)
clean = ev.load_clean_text(config.data_file("validation_set.txt"), config.ALPHABET)

TEXT_LEN, N = 512, 300
all_k = list(range(m.min_key_len, m.max_key_len + 1))
conf = np.zeros((len(all_k), len(all_k)), dtype=int)

with torch.no_grad():
    for ki, k in enumerate(all_k):
        batch = []
        for i in range(N):
            s = (i * TEXT_LEN * 13) % max(1, len(clean) - TEXT_LEN)
            key = ev.get_fixed_key(k, i, config.ALPHABET)
            cipher, _ = vinf.encrypt_vigenere(clean[s:s+TEXT_LEN], key, char2idx, idx2char)
            batch.append(vinf.text_to_tensor(cipher, char2idx, DEVICE))
        src = torch.stack(batch)
        unk = torch.full((N,), UNKNOWN_KEY, dtype=torch.long, device=DEVICE)
        _, _, keylen_logits = m(src, unk)
        for p in (keylen_logits.argmax(-1) + m.min_key_len).tolist():
            conf[ki][p - m.min_key_len] += 1

acc = conf.diagonal() / conf.sum(1) * 100
print(f"\nblind period accuracy at {TEXT_LEN} characters")
print("  " + "".join(f"{k:>7}" for k in all_k))
print("  " + "".join(f"{a:>7.1f}" for a in acc))
print(f"\nmean: {acc.mean():.1f}%")


## Frequency content of the position embeddings

Only meaningful for the learned absolute variant. The spectrum is normalised to
the noise floor, so a value of 1.0 means no structure at that frequency.

In [ ]:
import matplotlib.pyplot as plt

fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4.5))

if hasattr(m, "pos_emb"):
    P = m.pos_emb.weight.detach().cpu().float().numpy()[:512]
    P = P - P.mean(0, keepdims=True)
    spec = np.abs(np.fft.rfft(P, axis=0)).mean(1)
    freqs = np.fft.rfftfreq(512)
    norm = spec / np.median(spec[1:])

    print(f"{'period':>8} {'frequency':>10} {'power':>8}")
    for k in [2] + all_k:
        i = np.argmin(np.abs(freqs - 1.0 / k))
        print(f"{k:>8} {1/k:>10.4f} {norm[i]:>8.2f}" + ("  *" if norm[i] > 1.2 else ""))

    lo = np.argmin(np.abs(freqs - 1.0 / (m.max_key_len + 4)))
    a1.plot(freqs[lo:], norm[lo:], lw=1.4, color="#2563EB")
    ymax = a1.get_ylim()[1]
    for k in all_k:
        a1.axvline(1/k, ls="--", lw=0.5, color="gray")
        a1.text(1/k, ymax*0.96, str(k), fontsize=7, ha="center")
    a1.axvline(0.5, ls="--", lw=0.8, color="crimson")
    a1.text(0.5, ymax*0.96, "2", fontsize=7, ha="center", color="crimson")
    a1.set_xlabel("frequency (1/period)")
    a1.set_ylabel("power (relative to noise floor)")
    a1.set_title("Learned position embedding spectrum")
else:
    a1.text(0.5, 0.5, "no learned position embeddings\nin this variant",
            ha="center", va="center", fontsize=11)
    a1.set_axis_off()

pct = conf / conf.sum(1, keepdims=True) * 100
im = a2.imshow(pct, cmap="viridis", vmin=0, vmax=100, aspect="auto")
a2.set_xticks(range(len(all_k))); a2.set_xticklabels(all_k, fontsize=8)
a2.set_yticks(range(len(all_k))); a2.set_yticklabels(all_k, fontsize=8)
a2.set_xlabel("predicted k"); a2.set_ylabel("true k")
a2.set_title(f"Period estimation confusion ({TEXT_LEN} chars)")
for i in range(len(all_k)):
    for j in range(len(all_k)):
        if pct[i, j] > 5:
            a2.text(j, i, f"{pct[i,j]:.0f}", ha="center", va="center",
                    fontsize=7, color="white" if pct[i, j] < 60 else "black")
plt.colorbar(im, ax=a2, shrink=0.85)

plt.tight_layout()
plt.savefig(config.STORAGE_ROOT / f"encoding_analysis_{VARIANT}.png", dpi=150, bbox_inches="tight")
plt.show()


Power concentrates at a handful of short periods. Because structure at one
period also aligns positions separated by its multiples, the key lengths the
model resolves are the multiples of those periods; the lengths sharing no
divisor with any of them are the ones the confusion matrix shows it cannot
separate.